# CloudBQ pass 2b — the company object by foreign-key resolution (Vertex AI Workbench / Colab Enterprise)

Lane N of pass 2, extended to the **company**: every deal carries its L2 tier (`docs/PASS2_L2_TIERS.md`); this notebook resolves each deal
to its company through three foreign keys in order — the deal's own primary company association (`stg_deals.company_ids`), the ARR
clinic's final-customer map (`rec_three_layer`), an exact company-name match — and rolls the deals up to one row per sales region × company
× tier, the deals nested. The SQL is the repository's (`sql/company_tiers_body.sql`); it runs here twice: first as a **dry run** over the
snapshot passed as a query parameter (nothing written), then as the **views and snapshot tables** in BigQuery with a **scheduled refresh**.
The probe is this notebook. Same identity notes as `notebooks/README.md`; the token is read from Secret Manager or prompted, never stored.

## 0. Settings, constants, helpers

In [ ]:
import os, re, sys, json, shlex, pathlib, getpass, subprocess
from datetime import datetime, timezone
import numpy as np, pandas as pd
from IPython.display import display

_cwd = pathlib.Path.cwd()
REPO = _cwd.parent if (_cwd.name == "notebooks" and (_cwd.parent / ".git").exists()) else pathlib.Path.home() / "CloudBQ"
if not (REPO / ".git").exists():                                             # Colab Enterprise: clone the branch first
    subprocess.run(["git", "clone", "--branch", "claude/cloud-run-bigquery-pipeline-e94ziy", "https://github.com/Wkayaobama/CloudBQ.git", str(REPO)], check=True)
os.chdir(REPO)

def load_env(path):
    env = {}
    for line in path.read_text().splitlines():
        parts = shlex.split(line, comments=True)
        if parts and "=" in parts[0]:
            k, v = parts[0].split("=", 1); env[k] = v
    return env
ENV = load_env(REPO / ".env") if (REPO / ".env").exists() else {}
PROJECT = ENV.get("PROJECT") or os.environ.get("PROJECT") or input("GCP project: ")
DATASET = ENV.get("DATASET") or "hubspot"; BQ_LOCATION = ENV.get("BQ_LOCATION") or "EU"
SYNC_DATASET = ENV.get("SYNC_DATASET") or os.environ.get("SYNC_DATASET") or input("sync dataset (e.g. HubspotSync): ")
SYNC_COMPANIES_TABLE = ENV.get("SYNC_COMPANIES_TABLE") or os.environ.get("SYNC_COMPANIES_TABLE") or input("synced Companies table (e.g. Companies): ")
SYNC_COMPANY_ID_COLUMN = ENV.get("SYNC_COMPANY_ID_COLUMN") or "id"
ARR_PROJECT = ENV.get("ARR_PROJECT") or PROJECT; ARR_DATASET = ENV.get("ARR_DATASET") or "arrdash"
assert not PROJECT.startswith(("your-", "<")), "PROJECT still holds the placeholder"
PH = {"PROJECT": PROJECT, "DATASET": DATASET, "SYNC_DATASET": SYNC_DATASET, "SYNC_COMPANIES_TABLE": SYNC_COMPANIES_TABLE,
      "SYNC_COMPANY_ID_COLUMN": SYNC_COMPANY_ID_COLUMN, "ARR_PROJECT": ARR_PROJECT, "ARR_DATASET": ARR_DATASET}

def render(path):
    """The repository's SQL as sql/apply.sh renders it: `-- @include <file>` spliced with the sibling file, then the placeholders."""
    out = []
    for line in (REPO / path).read_text().splitlines():
        out.append(line)
        if line.startswith("-- @include "): out.extend((REPO / "sql" / line.split(" ", 2)[2].strip()).read_text().splitlines())
    text = "\n".join(out)
    for k, v in PH.items(): text = text.replace("${" + k + "}", v)
    assert "${" not in text, "unrendered placeholder"
    return text

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=REPO); print(r.stdout[-6000:]); print(r.stderr[-3000:], file=sys.stderr)
    assert r.returncode == 0, f"exit {r.returncode}"

K = json.loads((REPO / "scoring" / "deal_tiers_v1.constants.json").read_text())
EDGES = np.array(K["edges"], dtype=float); LABELS = K["labels"]; CLOSED = K["closed_labels"]; SOURCE = K["certainty_source"]

def bin_index(certainty):
    x = pd.to_numeric(certainty, errors="coerce")
    return pd.Series(np.digitize(x.fillna(-1.0), EDGES), index=x.index).astype("float").mask(x.isna())

def tier(certainty, is_closed, is_closed_won):
    idx = bin_index(certainty); out = idx.map(lambda i: None if pd.isna(i) else LABELS[int(i)])
    return out.mask(is_closed.fillna(False).astype(bool), CLOSED["lost"]).mask(is_closed_won.fillna(False).astype(bool), CLOSED["won"])
print("project", PROJECT, "| dataset", DATASET, "| sync", SYNC_DATASET, SYNC_COMPANIES_TABLE, "| arr", ARR_PROJECT, ARR_DATASET)

## 1. The snapshot — every deal through the function's HubSpot client, tiered as `deal_tiers_v1` would (lane N)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "cloud_function_new_deals" / "requirements.txt"), "db-dtypes", "pyarrow", "pandas", "numpy", "requests"], check=False)
sys.path.insert(0, str(REPO / "cloud_function_new_deals")); import main, requests
try:
    from google.cloud import secretmanager
    token = secretmanager.SecretManagerServiceClient().access_secret_version(name=f"projects/{PROJECT}/secrets/hubspot-token/versions/latest").payload.data.decode()
    print("token: Secret Manager hubspot-token")
except Exception as e:
    print(f"Secret Manager unavailable ({type(e).__name__}); prompting"); token = getpass.getpass("HubSpot private-app token (read scopes): ")
hs = main.HubSpot(token); del token
pipelines, stages = hs.pipelines()
main.PROPS = list(dict.fromkeys(main.PROPS + ["hs_is_closed", "hs_is_closed_won"]))     # icalps_oppocertainty, icalps_dealstatus, final_customer, sales_region are in PROPS
raw = hs.search_created(datetime(2000, 1, 1, tzinfo=timezone.utc), datetime.now(timezone.utc))   # HubSpot search caps at 10,000 results; the portal has ~2,000 deals
deals = pd.DataFrame([{"deal_id": str(d["id"]), **d.get("properties", {})} for d in raw])
for col in ("hs_deal_stage_probability", "icalps_oppocertainty", "amount_in_home_currency"):
    deals[col] = pd.to_numeric(deals[col], errors="coerce") if col in deals else np.nan
for col in ("hs_is_closed", "hs_is_closed_won"):
    deals[col] = deals[col].map({"true": True, "false": False}) if col in deals else False
for col in ("final_customer", "sales_region", "icalps_dealstatus", "dealname", "dealstage", "pipeline"):
    if col not in deals: deals[col] = None
deals["certainty_source"] = np.where(deals["pipeline"].isin([k for k in SOURCE if k != "default"]), deals["pipeline"].map(SOURCE), SOURCE["default"])
deals["certainty"] = np.where(deals["certainty_source"] == "icalps_oppocertainty", deals["icalps_oppocertainty"], deals["hs_deal_stage_probability"])
deals["bin_index"] = bin_index(deals["certainty"]); deals["tier"] = tier(deals["certainty"], deals["hs_is_closed"], deals["hs_is_closed_won"])
print(len(deals), "deals;", deals["pipeline"].map(pipelines).fillna(deals["pipeline"]).value_counts().to_dict())
display(deals.groupby(["pipeline", "icalps_dealstatus"], dropna=False).size().to_frame("deals"))

## 2. The dry run — the repository's SQL over the snapshot as a query parameter (0 BigQuery tables written)

`sql/company_tiers_body.sql` expects a `deal_source`; here it is `UNNEST(@deals_payload)` — the same CTE chain the views will run over the
sync. Two SELECTs: the deal level (`resolved`: one row per deal with its company and the resolver that found it) and the company object
(`rollup`). Both are exported as CSV; the job's statement type proves nothing was written.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location=BQ_LOCATION)
def S(name, typ, v): return bigquery.ScalarQueryParameter(name, typ, None if (v is None or (isinstance(v, float) and np.isnan(v))) else v)
payload = [bigquery.StructQueryParameter(None,
    S("deal_id", "STRING", str(r.deal_id)), S("pipeline_id", "STRING", r.pipeline), S("dealstage_id", "STRING", r.dealstage), S("dealname", "STRING", r.dealname),
    S("final_customer", "STRING", r.final_customer), S("sales_region", "STRING", r.sales_region), S("icalps_dealstatus", "STRING", r.icalps_dealstatus),
    S("certainty_source", "STRING", r.certainty_source), S("certainty", "FLOAT64", None if pd.isna(r.certainty) else float(r.certainty)),
    S("bin_index", "INT64", None if pd.isna(r.bin_index) else int(r.bin_index)), S("tier", "STRING", r.tier),
    S("is_closed", "BOOL", bool(r.hs_is_closed)), S("is_closed_won", "BOOL", bool(r.hs_is_closed_won)),
    S("amount_kusd", "FLOAT64", None if pd.isna(r.amount_in_home_currency) else float(r.amount_in_home_currency)))
    for r in deals.itertuples(index=False)]
body = render("sql/company_tiers_body.sql")
def dry(final_select):
    sql = "WITH deal_source AS (SELECT * FROM UNNEST(@deals_payload)),\n" + body + "\n" + final_select
    job = client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("deals_payload", "STRUCT", payload)]))
    df = job.result().to_dataframe()
    assert job.statement_type == "SELECT" and job.destination is None or str(job.destination).startswith("anon"), job.statement_type   # read-only
    print(f"job {job.job_id} | statement {job.statement_type} | bytes {job.total_bytes_processed} | slot ms {job.slot_millis} | 0 BigQuery tables written")
    return df
deal_level = dry("SELECT * EXCEPT (company_ids) FROM resolved ORDER BY region, company_id NULLS LAST, deal_id")
company_level = dry("SELECT * FROM rollup ORDER BY sales_region, company_id NULLS LAST, tier")
out_dir = pathlib.Path("/content") if pathlib.Path("/content").exists() else REPO / "notebooks" / "out"; out_dir.mkdir(exist_ok=True)
deal_level.to_csv(out_dir / "company_deals_preview.csv", index=False); company_level.to_csv(out_dir / "company_tiers_preview.csv", index=False)
print(f"exported {len(deal_level)} deal rows and {len(company_level)} company rows -> {out_dir}")
display(deal_level["resolver"].value_counts().to_frame("deals_by_resolver"))
display(deal_level.groupby("status_ok").size().to_frame("deals_by_icalps_dealstatus_rule"))
display(company_level.head(20))

## 3. Ship — the views, the snapshot tables and the refresh script's objects (`sql/company_tiers_v1.sql`, rendered here as `sql/apply.sh` renders it)

Nothing is written into the sync tables: `companies_synced_v1` is a view over the Companies asset, every derived column lives in the
`hubspot` dataset. Re-running is safe (`CREATE OR REPLACE` / `IF NOT EXISTS`).

In [ ]:
ship = render("sql/company_tiers_v1.sql")
job = client.query(ship); job.result()
print(f"job {job.job_id} | {ship.count('CREATE ')} statements")
display(client.query(f"SELECT table_name, table_type, creation_time FROM `{PROJECT}.{DATASET}.INFORMATION_SCHEMA.TABLES` WHERE table_name LIKE 'compan%' ORDER BY table_name").result().to_dataframe())

## 4. Equality — the dry run's rollup (the API snapshot) against the live view (the sync): same rule, two sources

In [ ]:
live = client.query(f"SELECT sales_region, entity_key, tier, company_id, resolver, n_deals, amount_kusd FROM `{PROJECT}.{DATASET}.company_tiers_v1`").result().to_dataframe()
eq = company_level[["sales_region", "entity_key", "tier", "n_deals"]].merge(live[["sales_region", "entity_key", "tier", "n_deals"]], on=["sales_region", "entity_key", "tier"], how="outer", suffixes=("_snapshot", "_sync"))
eq[["n_deals_snapshot", "n_deals_sync"]] = eq[["n_deals_snapshot", "n_deals_sync"]].fillna(0).astype(int); eq["diff"] = eq["n_deals_snapshot"] - eq["n_deals_sync"]
print(f"{len(eq)} company × tier groups; {int((eq['diff'] != 0).sum())} differ (deals created, edited or deleted between the sync at 02:00 UTC and this snapshot)")
display(eq[eq["diff"] != 0].head(20)); display(live["resolver"].value_counts().to_frame("live_groups_by_resolver"))

## 5. The refresh — register the scheduled query (paused by `DISABLE_SCHEDULE=1` until one manual refresh looked right), run it once, probe

`sql/company_tiers_schedule.sh` registers `company-tiers-refresh` (UTC, off the hour, after the 02:00 transfer) running
`sql/company_tiers_apply.sql`: the live view replaces the snapshot `company_tiers` inside one transaction, appends the slice to
`company_tiers_history` and the row to `company_tiers_refreshes` — like the raw root of pass 3, a past refresh stays reconstructible.

In [ ]:
sh("DRYRUN=1 bash sql/company_tiers_schedule.sh")
I_HAVE_READ_THE_DRY_RUN_AND_WANT_THE_REFRESH_REGISTERED = False
if I_HAVE_READ_THE_DRY_RUN_AND_WANT_THE_REFRESH_REGISTERED:
    sh("DISABLE_SCHEDULE=1 bash sql/company_tiers_schedule.sh")          # registered, not yet scheduled
    run_ts = datetime.now(timezone.utc).replace(microsecond=0)
    job = client.query(render("sql/company_tiers_apply.sql"), job_config=bigquery.QueryJobConfig(query_parameters=[bigquery.ScalarQueryParameter("run_time", "TIMESTAMP", run_ts)]))
    job.result(); print(f"manual refresh at {run_ts.isoformat()} — job {job.job_id}")

## 6. Probe — the refreshes, the snapshot, the summary every surface compares

In [ ]:
display(client.query(f"SELECT refresh_ts, orchestrator, status, error, n_rows, n_companies, n_unresolved, n_deals, duration_ms FROM `{PROJECT}.{DATASET}.company_tiers_refreshes` ORDER BY refresh_ts DESC LIMIT 5").result().to_dataframe())
summary = client.query(f"SELECT * FROM `{PROJECT}.{DATASET}.company_tiers_summary_v1` ORDER BY sales_region, tier, resolver").result().to_dataframe(); display(summary)
snap = client.query(f"SELECT refresh_ts, sales_region, entity_key, tier, company_id, company_name, resolver, n_deals, amount_kusd, total_pipeline_k FROM `{PROJECT}.{DATASET}.company_tiers` ORDER BY sales_region, company_id NULLS LAST, tier").result().to_dataframe()
snap.to_csv(out_dir / "company_tiers_snapshot.csv", index=False); print(f"snapshot: {len(snap)} rows -> {out_dir / 'company_tiers_snapshot.csv'}"); display(snap.head(20))

## 7. Enable — the daily schedule (only after a manual refresh and the probe looked right)

In [ ]:
I_HAVE_READ_THE_PROBE_AND_WANT_THE_DAILY_REFRESH = False
if I_HAVE_READ_THE_PROBE_AND_WANT_THE_DAILY_REFRESH:
    sh("bash sql/company_tiers_schedule.sh")                               # same config, auto-scheduling on; idempotent on re-run